# Suffix Arrays

A comprehensive guide to suffix arrays - a space-efficient alternative to suffix trees for string processing.

---

[< Previous: Aho-Corasick Algorithm](02_aho_corasick.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Suffix Trees >](04_suffix_trees.ipynb)

---

## 1. What is a Suffix Array?

A **suffix array** is a sorted array of all suffixes of a string. More precisely:

- Given a string `S` of length `n`
- The suffix array `SA` is an array of integers from `0` to `n-1`
- `SA[i]` = starting position of the `i`-th smallest suffix (in lexicographic order)

### Key Properties

1. **Space-efficient**: Uses O(n) space (just an array of integers), compared to O(n) with larger constants for suffix trees
2. **Enables fast pattern matching**: Binary search in O(m log n) time
3. **Foundation for many algorithms**: Used in bioinformatics, data compression, and text indexing

### Suffix Array Definition

```
SA[i] = starting position of the i-th smallest suffix in lexicographic order
```

This means if we sort all suffixes alphabetically, `SA[i]` tells us where the `i`-th suffix in that sorted order originally started in the string.

## ASCII Art Visualization: Suffix Array for "banana$"

```
All suffixes:
Index  Suffix
  0    banana$
  1    anana$
  2    nana$
  3    ana$
  4    na$
  5    a$
  6    $

Sorted suffixes:
Rank   Suffix      Original position (SA value)
  0    $           6
  1    a$          5
  2    ana$        3
  3    anana$      1
  4    banana$     0
  5    na$         4
  6    nana$       2

Suffix Array SA = [6, 5, 3, 1, 0, 4, 2]

Reading the array:
  SA[0] = 6  →  Smallest suffix starts at position 6: "$"
  SA[1] = 5  →  2nd smallest starts at position 5: "a$"
  SA[2] = 3  →  3rd smallest starts at position 3: "ana$"
  ...
```

## 2. Naive Construction - O(n² log n)

The simplest approach: generate all suffixes and sort them.

### Algorithm
1. Generate all n suffixes with their starting positions
2. Sort the suffixes lexicographically
3. Extract the starting positions

### Complexity Analysis
- Generating suffixes: O(n)
- Sorting: O(n log n) comparisons
- Each comparison: O(n) in worst case (comparing strings)
- **Total: O(n² log n)**

In [ ]:
def build_suffix_array_naive(text):
    """
    Build suffix array using naive O(n² log n) approach.
    
    Creates all suffixes, sorts them lexicographically, and returns
    an array of starting positions in sorted order.
    
    Parameters
    ----------
    text : str
        Input string (should end with a unique terminal character like '$')
    
    Returns
    -------
    list[int]
        Suffix array - SA[i] is the starting position of the i-th smallest suffix
    
    Examples
    --------
    >>> build_suffix_array_naive('banana$')
    [6, 5, 3, 1, 0, 4, 2]
    
    Time Complexity: O(n² log n)
    Space Complexity: O(n²) for storing all suffixes
    """
    n = len(text)
    # Create list of (suffix, starting_position) pairs
    suffixes = [(text[i:], i) for i in range(n)]
    # Sort by suffix string
    suffixes.sort(key=lambda x: x[0])
    # Extract starting positions
    return [pos for suffix, pos in suffixes]

In [ ]:
# Test naive construction
text = "banana$"
sa = build_suffix_array_naive(text)
print(f"Text: {text}")
print(f"Suffix Array: {sa}")
print()
print("Verification - suffixes in sorted order:")
for i, pos in enumerate(sa):
    print(f"  SA[{i}] = {pos}  →  '{text[pos:]}'") 

### Step-by-Step Naive Construction

In [ ]:
def build_suffix_array_naive_verbose(text):
    """
    Build suffix array with detailed step-by-step output.
    
    Parameters
    ----------
    text : str
        Input string
    
    Returns
    -------
    list[int]
        Suffix array
    """
    n = len(text)
    
    print("Step 1: Generate all suffixes")
    print("-" * 40)
    suffixes = []
    for i in range(n):
        suffix = text[i:]
        suffixes.append((suffix, i))
        print(f"  Position {i}: '{suffix}'")
    
    print()
    print("Step 2: Sort suffixes lexicographically")
    print("-" * 40)
    suffixes.sort(key=lambda x: x[0])
    for i, (suffix, pos) in enumerate(suffixes):
        print(f"  Rank {i}: '{suffix}' (originally at position {pos})")
    
    print()
    print("Step 3: Extract starting positions")
    print("-" * 40)
    sa = [pos for suffix, pos in suffixes]
    print(f"  Suffix Array SA = {sa}")
    
    return sa

sa = build_suffix_array_naive_verbose("banana$")

## 3. Manber-Myers Algorithm - O(n log n)

The Manber-Myers algorithm uses a clever **doubling technique** to achieve O(n log n) construction time.

### Key Idea
Instead of comparing entire suffixes (which can be O(n) per comparison), we:
1. First sort by the first character (k=1)
2. Then sort by the first 2 characters (k=2)
3. Then sort by the first 4 characters (k=4)
4. Continue doubling until k ≥ n

The trick: When sorting by 2k characters, we use the **ranks from the k-character sort** as keys. This makes each comparison O(1)!

### ASCII Art: Manber-Myers Doubling Process

```
Text: "banana$"

═══════════════════════════════════════════════════════════════════
Round 1 (k=1): Sort by first 1 character
═══════════════════════════════════════════════════════════════════
  Position:  0   1   2   3   4   5   6
  Character: b   a   n   a   n   a   $
  Rank:      2   1   3   1   3   1   0
                │   │   │   │   │   │
                └───┴───┴───┘   │   │
                  'a' gets rank 1   │
                                    │
                              '$' gets rank 0 (smallest)

═══════════════════════════════════════════════════════════════════
Round 2 (k=2): Sort by first 2 characters
═══════════════════════════════════════════════════════════════════
  Use pairs: (rank[i], rank[i+k]) where k=1
  
  Position 0: (rank[0], rank[1]) = (2, 1)  → "ba"
  Position 1: (rank[1], rank[2]) = (1, 3)  → "an"
  Position 2: (rank[2], rank[3]) = (3, 1)  → "na"
  Position 3: (rank[3], rank[4]) = (1, 3)  → "an"
  Position 4: (rank[4], rank[5]) = (3, 1)  → "na"
  Position 5: (rank[5], rank[6]) = (1, 0)  → "a$"
  Position 6: (rank[6], -)       = (0, -)  → "$"

═══════════════════════════════════════════════════════════════════
Round 3 (k=4): Sort by first 4 characters
═══════════════════════════════════════════════════════════════════
  Use pairs: (rank[i], rank[i+k]) where k=2
  Continue until all suffixes have unique ranks or k ≥ n

After log(n) rounds → Fully sorted suffix array!
```

In [ ]:
import collections

def build_suffix_array_manber_myers(text):
    """
    Build suffix array using Manber-Myers doubling algorithm.
    
    Uses recursive bucket sorting with doubling to achieve O(n log n)
    time complexity. Each round doubles the number of characters used
    for comparison.
    
    Parameters
    ----------
    text : str
        Input string (should end with a unique terminal character like '$')
    
    Returns
    -------
    list[int]
        Suffix array - SA[i] is the starting position of the i-th smallest suffix
    
    Examples
    --------
    >>> build_suffix_array_manber_myers('banana$')
    [6, 5, 3, 1, 0, 4, 2]
    
    Time Complexity: O(n log n)
    Space Complexity: O(n)
    """
    def sort_bucket(s, bucket, order):
        """
        Recursively sort a bucket of suffix indices.
        
        Parameters
        ----------
        s : str
            The input string
        bucket : iterable
            Indices to sort
        order : int
            Current prefix length for comparison (doubles each round)
        
        Returns
        -------
        list[int]
            Sorted indices
        """
        # Group indices by their prefix of length 'order'
        d = collections.defaultdict(list)
        for i in bucket:
            key = s[i:i + order]
            d[key].append(i)
        
        result = []
        # Process groups in sorted order
        for k, v in sorted(d.items()):
            if len(v) > 1:
                # Multiple suffixes share this prefix - recurse with doubled order
                result += sort_bucket(s, v, order * 2)
            else:
                # Single suffix - it's in its final position
                result.append(v[0])
        return result
    
    return sort_bucket(text, range(len(text)), 1)

In [ ]:
# Test Manber-Myers construction
text = "banana$"
sa_mm = build_suffix_array_manber_myers(text)
print(f"Text: {text}")
print(f"Suffix Array (Manber-Myers): {sa_mm}")
print()
print("Verification:")
for i, pos in enumerate(sa_mm):
    print(f"  SA[{i}] = {pos}  →  '{text[pos:]}'")

In [ ]:
def build_suffix_array_manber_myers_verbose(text):
    """
    Build suffix array with Manber-Myers algorithm showing each round.
    
    Parameters
    ----------
    text : str
        Input string
    
    Returns
    -------
    list[int]
        Suffix array
    """
    def sort_bucket(s, bucket, order, depth=0):
        indent = "  " * depth
        bucket = list(bucket)
        
        if depth == 0:
            print(f"\n{'='*60}")
            print(f"Round: Sorting by first {order} character(s)")
            print(f"{'='*60}")
        
        # Group by prefix
        d = collections.defaultdict(list)
        for i in bucket:
            key = s[i:i + order]
            d[key].append(i)
        
        print(f"{indent}Bucket {bucket} grouped by prefix (len={order}):")
        for k, v in sorted(d.items()):
            print(f"{indent}  '{k}': positions {v}")
        
        result = []
        for k, v in sorted(d.items()):
            if len(v) > 1:
                print(f"{indent}  → '{k}' has {len(v)} suffixes, need to recurse with order={order*2}")
                result += sort_bucket(s, v, order * 2, depth + 1)
            else:
                print(f"{indent}  → '{k}' unique, position {v[0]} is final")
                result.append(v[0])
        
        return result
    
    print(f"Text: '{text}'")
    print(f"Length: {len(text)}")
    sa = sort_bucket(text, range(len(text)), 1)
    print(f"\nFinal Suffix Array: {sa}")
    return sa

sa = build_suffix_array_manber_myers_verbose("banana$")

### Alternative Manber-Myers Implementation (Iterative with Ranks)

In [ ]:
def build_suffix_array_manber_myers_iterative(text):
    """
    Iterative Manber-Myers suffix array construction.
    
    Uses rank arrays and pair comparisons for efficient sorting.
    
    Parameters
    ----------
    text : str
        Input string
    
    Returns
    -------
    list[int]
        Suffix array
    
    Time Complexity: O(n log n)
    Space Complexity: O(n)
    """
    n = len(text)
    if n == 0:
        return []
    if n == 1:
        return [0]
    
    # Initialize suffix array and rank array
    sa = list(range(n))
    rank = [ord(c) for c in text]
    tmp = [0] * n
    
    k = 1
    while k < n:
        # Sort by (rank[i], rank[i+k]) pairs
        def compare_key(i):
            return (rank[i], rank[i + k] if i + k < n else -1)
        
        sa.sort(key=compare_key)
        
        # Compute new ranks
        tmp[sa[0]] = 0
        for i in range(1, n):
            prev_key = compare_key(sa[i - 1])
            curr_key = compare_key(sa[i])
            tmp[sa[i]] = tmp[sa[i - 1]] + (1 if curr_key > prev_key else 0)
        
        rank, tmp = tmp, rank
        
        # Early termination if all ranks are unique
        if rank[sa[n - 1]] == n - 1:
            break
        
        k *= 2
    
    return sa

# Test
text = "banana$"
sa = build_suffix_array_manber_myers_iterative(text)
print(f"Text: {text}")
print(f"Suffix Array: {sa}")

## 4. Pattern Search using Suffix Array - O(m log n)

Once we have a suffix array, we can search for patterns using **binary search**.

### Key Insight
- If pattern P occurs in text T, it must be a **prefix** of some suffix
- Suffixes are sorted in the suffix array
- Binary search finds where P would fit in the sorted order

### ASCII Art: Pattern Search with Binary Search

```
Text = "banana$"
SA = [6, 5, 3, 1, 0, 4, 2]
Find: "ana"

Sorted suffixes (what SA represents):
  SA[0]=6 → "$"
  SA[1]=5 → "a$"
  SA[2]=3 → "ana$"      ← "ana" is prefix! ✓
  SA[3]=1 → "anana$"    ← "ana" is prefix! ✓
  SA[4]=0 → "banana$"
  SA[5]=4 → "na$"
  SA[6]=2 → "nana$"

Binary search for lower bound of "ana":
  mid = 3 → "anana$"
  Compare: "ana" vs "anana$"[:3] = "ana" → equal, but need lower bound
  Search left...
  
Binary search for upper bound of "ana":
  Find first suffix where "ana" is NOT a prefix
  
Result: All occurrences in range [2, 4) of SA
  → Positions SA[2]=3 and SA[3]=1
  → Pattern "ana" found at positions: 1, 3
```

In [ ]:
def search_pattern(text, sa, pattern):
    """
    Search for pattern occurrences using suffix array and binary search.
    
    Finds all positions where the pattern occurs as a substring of text.
    Uses binary search to find the range of suffixes that start with pattern.
    
    Parameters
    ----------
    text : str
        The text to search in
    sa : list[int]
        Suffix array of the text
    pattern : str
        Pattern to search for
    
    Returns
    -------
    list[int]
        Sorted list of positions where pattern occurs
    
    Examples
    --------
    >>> text = "banana$"
    >>> sa = build_suffix_array_naive(text)
    >>> search_pattern(text, sa, "ana")
    [1, 3]
    
    Time Complexity: O(m log n) where m = len(pattern), n = len(text)
    Space Complexity: O(1) additional space
    """
    n = len(text)
    m = len(pattern)
    
    if m == 0:
        return list(range(n))
    
    # Binary search for lower bound
    # Find first suffix that is >= pattern
    left, right = 0, n
    while left < right:
        mid = (left + right) // 2
        suffix = text[sa[mid]:]
        if suffix[:m] < pattern:
            left = mid + 1
        else:
            right = mid
    lower = left
    
    # Binary search for upper bound
    # Find first suffix that is > pattern (pattern is not a prefix)
    left, right = 0, n
    while left < right:
        mid = (left + right) // 2
        suffix = text[sa[mid]:]
        if suffix[:m] <= pattern:
            left = mid + 1
        else:
            right = mid
    upper = left
    
    # Extract and sort the positions
    return sorted([sa[i] for i in range(lower, upper)])

In [ ]:
# Test pattern search
text = "banana$"
sa = build_suffix_array_naive(text)

patterns = ["ana", "na", "a", "ban", "xyz", "$"]

print(f"Text: '{text}'")
print(f"Suffix Array: {sa}")
print()

for pattern in patterns:
    positions = search_pattern(text, sa, pattern)
    print(f"Pattern '{pattern}': found at positions {positions}")
    if positions:
        for pos in positions:
            print(f"    text[{pos}:{pos+len(pattern)}] = '{text[pos:pos+len(pattern)]}'")

In [ ]:
def search_pattern_verbose(text, sa, pattern):
    """
    Search for pattern with detailed binary search visualization.
    
    Parameters
    ----------
    text : str
        The text to search in
    sa : list[int]
        Suffix array of the text
    pattern : str
        Pattern to search for
    
    Returns
    -------
    list[int]
        Positions where pattern occurs
    """
    n = len(text)
    m = len(pattern)
    
    print(f"Searching for '{pattern}' in '{text}'")
    print(f"Suffix Array: {sa}")
    print()
    print("Sorted suffixes:")
    for i, pos in enumerate(sa):
        suffix = text[pos:]
        prefix_match = "✓" if suffix[:m] == pattern else " "
        print(f"  SA[{i}]={pos} → '{suffix}' {prefix_match}")
    print()
    
    # Lower bound search
    print("Binary search for LOWER bound:")
    left, right = 0, n
    while left < right:
        mid = (left + right) // 2
        suffix = text[sa[mid]:]
        cmp = suffix[:m]
        print(f"  left={left}, right={right}, mid={mid}")
        print(f"    SA[{mid}]={sa[mid]} → comparing '{cmp}' vs '{pattern}'")
        if cmp < pattern:
            print(f"    '{cmp}' < '{pattern}' → search right")
            left = mid + 1
        else:
            print(f"    '{cmp}' >= '{pattern}' → search left")
            right = mid
    lower = left
    print(f"  Lower bound: {lower}")
    print()
    
    # Upper bound search
    print("Binary search for UPPER bound:")
    left, right = 0, n
    while left < right:
        mid = (left + right) // 2
        suffix = text[sa[mid]:]
        cmp = suffix[:m]
        print(f"  left={left}, right={right}, mid={mid}")
        print(f"    SA[{mid}]={sa[mid]} → comparing '{cmp}' vs '{pattern}'")
        if cmp <= pattern:
            print(f"    '{cmp}' <= '{pattern}' → search right")
            left = mid + 1
        else:
            print(f"    '{cmp}' > '{pattern}' → search left")
            right = mid
    upper = left
    print(f"  Upper bound: {upper}")
    print()
    
    print(f"Range: [{lower}, {upper})")
    positions = sorted([sa[i] for i in range(lower, upper)])
    print(f"Positions: {positions}")
    return positions

_ = search_pattern_verbose("banana$", sa, "ana")

## 5. LCP Array - Longest Common Prefix

The **LCP Array** stores the length of the longest common prefix between consecutive suffixes in the sorted order.

### Definition
```
LCP[i] = length of longest common prefix between suffix SA[i-1] and suffix SA[i]
LCP[0] = 0 (or undefined, no previous suffix)
```

### Applications
1. **Longest Repeated Substring**: Maximum value in LCP array
2. **Number of distinct substrings**: n(n+1)/2 - sum(LCP)
3. **Efficient string comparison**: Range minimum queries on LCP

### ASCII Art: LCP Array for "banana$"

```
Suffix Array SA = [6, 5, 3, 1, 0, 4, 2]

i    SA[i]   Suffix      LCP[i]   Explanation
─────────────────────────────────────────────────────────────
0    6       $           0        (no previous suffix)
1    5       a$          0        LCP("$", "a$") = 0
2    3       ana$        1        LCP("a$", "ana$") = 1 ("a")
3    1       anana$      3        LCP("ana$", "anana$") = 3 ("ana")
4    0       banana$     0        LCP("anana$", "banana$") = 0
5    4       na$         0        LCP("banana$", "na$") = 0
6    2       nana$       2        LCP("na$", "nana$") = 2 ("na")

LCP Array = [0, 0, 1, 3, 0, 0, 2]

Longest Repeated Substring:
  max(LCP) = 3 at position 3
  Suffix at SA[3]=1: "anana$"
  LRS = "ana" (first 3 characters)
```

In [ ]:
def build_lcp_array(text, sa):
    """
    Build LCP (Longest Common Prefix) array from text and suffix array.
    
    Uses Kasai's algorithm for O(n) construction.
    LCP[i] = length of longest common prefix between suffixes at SA[i-1] and SA[i].
    
    Parameters
    ----------
    text : str
        The input text
    sa : list[int]
        Suffix array of the text
    
    Returns
    -------
    list[int]
        LCP array where LCP[i] is the longest common prefix length
        between suffix SA[i-1] and suffix SA[i]. LCP[0] = 0.
    
    Examples
    --------
    >>> text = "banana$"
    >>> sa = build_suffix_array_naive(text)
    >>> build_lcp_array(text, sa)
    [0, 0, 1, 3, 0, 0, 2]
    
    Time Complexity: O(n)
    Space Complexity: O(n)
    """
    n = len(text)
    if n == 0:
        return []
    
    # Build inverse suffix array (rank array)
    # rank[i] = position of suffix starting at i in the sorted order
    rank = [0] * n
    for i in range(n):
        rank[sa[i]] = i
    
    lcp = [0] * n
    k = 0  # Current LCP length
    
    # Process suffixes in text order (Kasai's algorithm)
    for i in range(n):
        if rank[i] == 0:
            k = 0
            continue
        
        # Previous suffix in sorted order
        j = sa[rank[i] - 1]
        
        # Extend LCP
        while i + k < n and j + k < n and text[i + k] == text[j + k]:
            k += 1
        
        lcp[rank[i]] = k
        
        # Key insight: LCP can decrease by at most 1 when moving to next suffix
        if k > 0:
            k -= 1
    
    return lcp

In [ ]:
def build_lcp_array_naive(text, sa):
    """
    Build LCP array using naive O(n²) approach.
    
    Simpler but slower - compares each pair of adjacent suffixes directly.
    
    Parameters
    ----------
    text : str
        The input text
    sa : list[int]
        Suffix array of the text
    
    Returns
    -------
    list[int]
        LCP array
    
    Time Complexity: O(n²)
    Space Complexity: O(n)
    """
    n = len(text)
    lcp = [0] * n
    
    for i in range(1, n):
        # Compare suffix at SA[i-1] with suffix at SA[i]
        s1 = text[sa[i - 1]:]
        s2 = text[sa[i]:]
        
        # Count common prefix length
        j = 0
        while j < len(s1) and j < len(s2) and s1[j] == s2[j]:
            j += 1
        lcp[i] = j
    
    return lcp

In [ ]:
# Test LCP array construction
text = "banana$"
sa = build_suffix_array_naive(text)
lcp = build_lcp_array(text, sa)

print(f"Text: '{text}'")
print(f"Suffix Array: {sa}")
print(f"LCP Array: {lcp}")
print()
print("Detailed view:")
print(f"{'i':>3} {'SA[i]':>6} {'Suffix':<12} {'LCP[i]':>6} Common Prefix")
print("-" * 50)
for i in range(len(sa)):
    suffix = text[sa[i]:]
    if i > 0 and lcp[i] > 0:
        common = suffix[:lcp[i]]
    else:
        common = ""
    print(f"{i:>3} {sa[i]:>6} {suffix:<12} {lcp[i]:>6} '{common}'")

### Application: Longest Repeated Substring

In [ ]:
def longest_repeated_substring(text):
    """
    Find the longest substring that appears at least twice.
    
    Uses suffix array and LCP array to find the maximum LCP value.
    
    Parameters
    ----------
    text : str
        Input text
    
    Returns
    -------
    tuple[str, list[int]]
        (longest repeated substring, list of positions where it occurs)
    
    Examples
    --------
    >>> longest_repeated_substring("banana$")
    ('ana', [1, 3])
    
    Time Complexity: O(n log n) for SA construction + O(n) for LCP
    Space Complexity: O(n)
    """
    if len(text) <= 1:
        return ("", [])
    
    sa = build_suffix_array_manber_myers(text)
    lcp = build_lcp_array(text, sa)
    
    # Find maximum LCP
    max_lcp = 0
    max_idx = 0
    for i in range(len(lcp)):
        if lcp[i] > max_lcp:
            max_lcp = lcp[i]
            max_idx = i
    
    if max_lcp == 0:
        return ("", [])
    
    # Extract the repeated substring
    lrs = text[sa[max_idx]:sa[max_idx] + max_lcp]
    
    # Find all occurrences
    positions = search_pattern(text, sa, lrs)
    
    return (lrs, positions)

# Test
test_cases = [
    "banana$",
    "mississippi$",
    "abracadabra$",
    "aaaaaa$"
]

for text in test_cases:
    lrs, positions = longest_repeated_substring(text)
    print(f"Text: '{text}'")
    print(f"  Longest repeated substring: '{lrs}'")
    print(f"  Occurs at positions: {positions}")
    print()

## Complexity Comparison

| Operation | Naive | Manber-Myers | Notes |
|-----------|-------|--------------|-------|
| **Construction** | O(n² log n) | O(n log n) | Manber-Myers uses doubling trick |
| **Space** | O(n²)* | O(n) | *Naive stores all suffixes temporarily |
| **Pattern Search** | O(m log n) | O(m log n) | Same - uses binary search |
| **LCP Construction** | O(n²) | O(n) | Kasai's algorithm is linear |

Where:
- n = length of text
- m = length of pattern

## Performance Comparison

In [ ]:
import time
import random
import string

def generate_random_text(n, alphabet_size=4):
    """Generate random text of length n with given alphabet size."""
    alphabet = string.ascii_lowercase[:alphabet_size]
    return ''.join(random.choice(alphabet) for _ in range(n)) + '$'

def benchmark_construction(sizes, num_trials=3):
    """
    Benchmark suffix array construction algorithms.
    
    Parameters
    ----------
    sizes : list[int]
        Text sizes to benchmark
    num_trials : int
        Number of trials per size
    
    Returns
    -------
    dict
        Timing results
    """
    results = {'sizes': sizes, 'naive': [], 'manber_myers': []}
    
    for n in sizes:
        naive_times = []
        mm_times = []
        
        for _ in range(num_trials):
            text = generate_random_text(n)
            
            # Benchmark naive (skip for large n)
            if n <= 5000:
                start = time.perf_counter()
                sa_naive = build_suffix_array_naive(text)
                naive_times.append(time.perf_counter() - start)
            
            # Benchmark Manber-Myers
            start = time.perf_counter()
            sa_mm = build_suffix_array_manber_myers(text)
            mm_times.append(time.perf_counter() - start)
        
        results['naive'].append(sum(naive_times) / len(naive_times) if naive_times else None)
        results['manber_myers'].append(sum(mm_times) / len(mm_times))
        
        print(f"n={n:>6}: Naive={results['naive'][-1]:.4f}s" if results['naive'][-1] else f"n={n:>6}: Naive=skipped", 
              f", Manber-Myers={results['manber_myers'][-1]:.4f}s")
    
    return results

# Run benchmark
print("Benchmarking Suffix Array Construction")
print("=" * 50)
sizes = [100, 500, 1000, 2000, 5000]
results = benchmark_construction(sizes)

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

# Plot naive times (only where available)
naive_sizes = [s for s, t in zip(results['sizes'], results['naive']) if t is not None]
naive_times = [t for t in results['naive'] if t is not None]
if naive_times:
    ax.plot(naive_sizes, naive_times, 'o-', label='Naive O(n² log n)', linewidth=2, markersize=8)

# Plot Manber-Myers times
ax.plot(results['sizes'], results['manber_myers'], 's-', label='Manber-Myers O(n log n)', 
        linewidth=2, markersize=8)

ax.set_xlabel('Text Length (n)', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Suffix Array Construction: Naive vs Manber-Myers', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

## Complete Example: Text Analysis

In [ ]:
def analyze_text(text):
    """
    Comprehensive text analysis using suffix array and LCP array.
    
    Parameters
    ----------
    text : str
        Input text (should end with unique terminal character)
    """
    print(f"Text: '{text}'")
    print(f"Length: {len(text)}")
    print("=" * 60)
    
    # Build suffix array
    sa = build_suffix_array_manber_myers(text)
    print(f"\nSuffix Array: {sa}")
    
    # Build LCP array
    lcp = build_lcp_array(text, sa)
    print(f"LCP Array: {lcp}")
    
    # Display sorted suffixes with LCP
    print("\nSorted Suffixes:")
    print(f"{'Rank':>4} {'Pos':>4} {'LCP':>4}  Suffix")
    print("-" * 40)
    for i in range(len(sa)):
        print(f"{i:>4} {sa[i]:>4} {lcp[i]:>4}  {text[sa[i]:]}")
    
    # Find longest repeated substring
    lrs, positions = longest_repeated_substring(text)
    print(f"\nLongest Repeated Substring: '{lrs}'")
    print(f"Occurs at positions: {positions}")
    
    # Count distinct substrings
    n = len(text)
    total_substrings = n * (n + 1) // 2
    duplicates = sum(lcp)
    distinct = total_substrings - duplicates
    print(f"\nDistinct Substrings: {distinct}")
    print(f"  (Total possible: {total_substrings}, Duplicates: {duplicates})")

# Analyze example texts
analyze_text("banana$")
print("\n" + "=" * 60 + "\n")
analyze_text("mississippi$")

## Summary

### Key Takeaways

1. **Suffix Array** is a space-efficient alternative to suffix trees
   - Stores just n integers instead of complex tree structure
   - Enables same pattern matching capabilities

2. **Construction Algorithms**
   - Naive: O(n² log n) - simple but slow
   - Manber-Myers: O(n log n) - uses doubling technique
   - (Other algorithms exist: DC3/Skew achieves O(n))

3. **Pattern Search**
   - Binary search in O(m log n) time
   - Can be improved to O(m + log n) with LCP array

4. **LCP Array**
   - Stores longest common prefix between adjacent sorted suffixes
   - Enables finding longest repeated substring in O(n) time
   - Can count distinct substrings efficiently

### When to Use Suffix Arrays

- When you need to search for multiple patterns in the same text
- When memory is constrained (more compact than suffix trees)
- In bioinformatics for genome analysis
- In data compression algorithms
- For plagiarism detection and document similarity

---

## Exercises

### Exercise 1: K-mer Occurrence Counting with a Suffix Array (*)

Suffix arrays support binary-search-based pattern lookup in O(m log n) time. This is the backbone of modern short-read aligners like BWA.

**Task:** Implement `count_kmer_occurrences(sequence, k)` that builds a suffix array for the given DNA sequence and then counts how many times each distinct k-mer appears, using binary search (as in `search_pattern`). Report the top 5 most frequent k-mers.

Hint: iterate over all distinct k-mers from the suffix array itself — adjacent entries in the SA with matching k-length prefixes belong to the same k-mer.

In [ ]:
DNA = "ATCGATCGATCGAATTCCGATCGATCGATCG$"


def count_kmer_occurrences(sequence: str, k: int) -> list[tuple[str, int]]:
    """
    Count occurrences of every distinct k-mer using the suffix array.

    Args:
        sequence: DNA string (should end with '$' sentinel)
        k: k-mer length

    Returns:
        List of (kmer, count) pairs sorted by descending count, then kmer
    """
    # YOUR CODE HERE
    # Hint: call build_suffix_array_naive(sequence) then use search_pattern
    pass


# counts = count_kmer_occurrences(DNA, k=4)
# print("Top 5 most frequent 4-mers:")
# for kmer, count in counts[:5]:
#     print(f"  {kmer}: {count}")

### Exercise 2: Longest Repeated Motif via Suffix Array + LCP (**)

The LCP (Longest Common Prefix) array lets us find the longest repeated substring in O(n) time: the answer is simply `max(lcp)`.

**Task:** Implement `find_longest_repeated_motif(sequence)` using the suffix array and LCP array. Return the repeated substring and all positions where it occurs. Test on the genomic sequence below and compare your result to `SuffixTreeLRS` from the suffix trees notebook (same answer expected).

Recall: if `lcp[i] = L`, then `SA[i-1]` and `SA[i]` share a common prefix of length `L`.

In [ ]:
GENOMIC = "ATCGATCGATCGAATTCCGATCGATCGATCG$"


def find_longest_repeated_motif(sequence: str) -> tuple[str, list[int]]:
    """
    Find the longest substring that appears at least twice.

    Uses the suffix array and LCP array:
      - Build SA with build_suffix_array_naive(sequence)
      - Build LCP with build_lcp_array(sequence, sa)
      - The maximum LCP value identifies the repeated motif
      - Collect all SA positions where that LCP length is achieved

    Args:
        sequence: DNA string ending with '$'

    Returns:
        Tuple of (repeated_substring, sorted list of start positions)
    """
    # YOUR CODE HERE
    pass


# motif, positions = find_longest_repeated_motif(GENOMIC)
# print(f"Longest repeated motif: '{motif}' (length {len(motif)})")
# print(f"Occurs at positions: {positions}")
# for p in positions:
#     print(f"  pos {p}: {GENOMIC[p:p+len(motif)]}")

---

[< Previous: Aho-Corasick Algorithm](02_aho_corasick.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Suffix Trees >](04_suffix_trees.ipynb)

---